# rsl-coder-7b -- all phases

Self-contained. **Upload nothing else.**

1. Run **cell 1** (setup) then **cell 2** (unpack + selftest)
2. Wait for `HARNESS VERIFIED`
3. Run **cell 3** (status)
4. Run the phase cells in order

**Do not use `Runtime -> Run all` for phases 3 and beyond.** It
would start the 2-4 hour phases back to back. The gates stop a
broken chain, but you would rather see each result before starting
the next.

Embedded: 21 files.


## Setup


In [ ]:
# ============================================================================
# CELL 1 -- SETUP: dependencies, directories, Drive     (no model download)
# ============================================================================
import base64, json, os, subprocess, sys, traceback
from pathlib import Path

# --- dependencies ----------------------------------------------------------
# BUG THIS CELL HAD: the pip install was dropped when setup and payload were
# split into two cells. Colab ships no bitsandbytes, so 4-bit loading died with
# "requires bitsandbytes>=0.46.1" AFTER the 15GB config/tokenizer download.
# bitsandbytes is deliberately NOT pinned: the pin drifted below the minimum
# transformers now demands, and the version that matters is dictated by
# transformers, not by us.
print("installing dependencies...")
_rc = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q",
     "bitsandbytes>=0.46.1", "accelerate>=0.34", "peft>=0.13"],
    capture_output=True, text=True)
print(_rc.stdout[-400:] or "")
if _rc.returncode:
    print("pip stderr:", _rc.stderr[-800:])
else:
    import bitsandbytes, accelerate, peft
    print(f"bitsandbytes {bitsandbytes.__version__}")
    print(f"accelerate {accelerate.__version__}")
    print(f"peft {peft.__version__}")
import transformers
print(f"transformers {transformers.__version__}")

RSL = "/content/rsl"
for d in ("", "refs", "phases"):
    Path(RSL + "/" + d if d else RSL).mkdir(parents=True, exist_ok=True)

REPORT = {"stage": "setup", "ok": None, "error": None, "tb": None, "steps": []}

# Drive is best-effort. Pre-emption means results must persist, but a mount
# failure must not kill the run -- that would waste the whole session.
DRIVE_ROOT = None
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    DRIVE_ROOT = Path("/content/drive/MyDrive/rsl_model")
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
    print(f"drive ok -> {DRIVE_ROOT}")
except Exception as e:
    print(f"WARNING: no Drive ({e}). Results live only in this session and "
          f"will be LOST if it ends.")


def step(name, **kw):
    REPORT["steps"].append(dict(name=name, **kw))
    print(f"[step] {name}", flush=True)


def flush():
    """Write the report locally always, and to Drive when available."""
    blob = json.dumps(REPORT, indent=2)
    try:
        Path(RSL, "report.json").write_text(blob)
    except Exception:
        pass
    if DRIVE_ROOT:
        try:
            (DRIVE_ROOT / "report.json").write_text(blob)
        except Exception as e:
            print(f"  (drive write failed: {e})")
    print(f"[report] stage={REPORT['stage']} ok={REPORT['ok']}", flush=True)


sys.path.insert(0, RSL)
print("\ncell 1 ready")


## Unpack + selftest

Downloads **nothing**. Scores the harness against 10 reference
solutions and 10 logic-bugged ones with a stub model. If this
aborts, the reward signal is broken and a 15GB download would only
prove it. ~10 seconds.


In [ ]:
# ============================================================================
# CELL 2 -- UNPACK + SELFTEST     (~10s, no model, no download)
# All source files are embedded below as zlib+base64. Nothing else to upload.
# ============================================================================
import base64, json, sys, traceback, zlib
from pathlib import Path

RSL = "/content/rsl"

# zlib then base64, reversed here. Python source compresses ~4x, which is what
# keeps this cell a manageable size instead of the 155 KB of raw base64.
_BLOB = "..."
_files = json.loads(zlib.decompress(base64.b64decode(_BLOB)).decode())
for _rel, _b64 in _files.items():
    _dest = Path(RSL) / _rel
    _dest.parent.mkdir(parents=True, exist_ok=True)
    _dest.write_bytes(base64.b64decode(_b64))
print(f"unpacked {len(_files)} files into {RSL}")
sys.path.insert(0, RSL)

REPORT["stage"] = "selftest"
step("selftest")
try:
    import rsl_harness
    passed, fails = rsl_harness.self_test(verbose=True)
    REPORT["selftest"] = [passed, fails]
    if not passed:
        REPORT.update(ok=False, error="harness self-test failed",
                      tb="\n".join(fails))
        flush()
        raise SystemExit("HARNESS BROKEN -- do not download a model")
except SystemExit:
    flush()
    raise
except Exception:
    REPORT.update(ok=False, error="selftest raised",
                  tb=traceback.format_exc())
    flush()
    raise

REPORT["ok"] = True
flush()
print("\nHARNESS VERIFIED. Run CELL 3, then the phase cells in order.")


## Status

Which phases have run and whether their gates passed.


In [ ]:
# ============================================================================
# CELL 3 -- STATUS: which phases have run and whether their gates passed
# ============================================================================
from runner import print_table, phases_table, all_done

print_table()
print()
print("Run the phase cells below in order. Each refuses to start if the")
print("previous phase did not pass its gate.")
if all_done():
    print("\nall phases have run.")


## Phase 0 -- baseline

Loads the 7B base in 4-bit and scores 10 tasks with real unit tests.
**Expect 7-9/10.** This is the baseline every later phase is measured against, so it is not optional. ~15 min including the download.


In [ ]:
# --- phase 0 ---
# --- phase 0: baseline ---
from runner import run_phase
run_phase(0, REPORT)


## Phase 1 -- abliteration

Finds each layer's refusal direction from dual-use vs benign prompts and
projects it out of the residual stream. **Expect 0% refusal, <=2 tasks lost.** ~5 min.


In [ ]:
# --- phase 1 ---
# --- phase 1: abliteration ---
from runner import run_phase
run_phase(1, REPORT)


## Phase 2 -- fisher rank table

Calibration only, no training. Produces the per-layer LoRA rank table from
the eFIM diagonal on LoRA-B. Uniform rank is a design flaw; this is the
fix. Score LoRA-B only -- at init B=0 so dL/dA is exactly zero. ~15 min.


In [ ]:
# --- phase 2 ---
# --- phase 2: fisher rank table ---
from runner import run_phase
run_phase(2, REPORT)


## Phase 3 -- QLoRA reference

**Mandatory.** The reference QLoRA run at fixed rank. Nothing after this
is measurable without it. **2-4 hours.**


In [ ]:
# --- phase 3 ---
# --- phase 3: QLoRA reference ---
from runner import run_phase
run_phase(3, REPORT)


## Phase 4 -- elastic MoE

Three elasticity mechanisms: MoE experts that grow and are pruned (Net2Net
init, so growing costs no accuracy), rank reallocated mid-training, and
per-parameter LR. **Several hours.**


In [ ]:
# --- phase 4 ---
# --- phase 4: elastic MoE ---
from runner import run_phase
run_phase(4, REPORT)


## Phase 5 -- autonomous harvester

arXiv -> clone -> **execute**. The information supply, and the thing that
stops the loop plateauing. Keep only code that ran and beat the baseline.
**Expect ~5% yield; that is normal, not a bug.** ~1 hour.


In [ ]:
# --- phase 5 ---
# --- phase 5: autonomous harvester ---
from runner import run_phase
run_phase(5, REPORT)


## Phase 6 -- self-play + RSL

The loop: the model writes tasks, execution decides, it trains on the
winners, then rewrites its own harness. **Multi-hour.** Expect a plateau
at 4-6 rounds; that is the correct outcome, not a failure.


In [ ]:
# --- phase 6 ---
# --- phase 6: self-play + RSL ---
from runner import run_phase
run_phase(6, REPORT)


## Phase 7 -- finalize

Re-abliterate (SFT reinstalls refusals), merge to fp16, write a model card
containing only measured numbers. ~30 min.


In [ ]:
# --- phase 7 ---
# --- phase 7: finalize ---
from runner import run_phase
run_phase(7, REPORT)
